In [3]:
import os
import cv2
import random
import pandas as pd
from collections import defaultdict

# =========================
# USER INPUT PATHS
# =========================

source_path = r"/mnt/lustre/lalit47/us_data/bench/Carotid Ultrasound Boundary Study (CUBS)"
images_folder = os.path.join(source_path, "IMAGES")
csv_path = os.path.join(source_path, "ClinicalDatabase-CUBS.csv")

target_path = r"/mnt/lustre/lalit47/down/class/CUBS"

train_ratio = 0.7
val_ratio = 0.1
test_ratio = 0.2

random_seed = 42
random.seed(random_seed)

image_size = (256, 256)


def process_and_save(src, dst):
    im = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if im is None:
        print(f"Failed to read: {src}")
        return False

    im = cv2.resize(im, image_size, interpolation=cv2.INTER_LINEAR)
    cv2.imwrite(dst, im)
    return True


def normalize_id(x):
    """
    Convert Patient ID to a clean string.
    Example:
    12.0 -> "12"
    "P001 " -> "P001"
    """
    if pd.isna(x):
        return None

    x = str(x).strip()

    if x.endswith(".0"):
        x = x[:-2]

    return x


def collect_classification_images():
    data_by_class = defaultdict(list)

    df = pd.read_csv(
        csv_path,
        sep=";",
        decimal=",",
        engine="python"
    )

    # Remove fully empty columns caused by leading semicolon
    df = df.dropna(axis=1, how="all")

    print("CSV columns:")
    print(df.columns.tolist())

    patient_id_col = "Patient ID"
    label_col = "FUPEvents"

    df[patient_id_col] = df[patient_id_col].apply(normalize_id)

    patient_to_label = {}

    for _, row in df.iterrows():
        patient_id = row[patient_id_col]
        label = row[label_col]

        if patient_id is None:
            continue

        if pd.isna(label):
            continue

        label = int(label)

        if label not in [0, 1]:
            continue

        patient_to_label[patient_id] = label

    print(f"Total patients in CSV: {len(patient_to_label)}")

    image_files = [
        f for f in os.listdir(images_folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"))
    ]

    matched_count = 0
    unmatched_count = 0

    for file_name in image_files:
        image_path = os.path.join(images_folder, file_name)
        base_name = os.path.splitext(file_name)[0]

        matched_patient_id = None

        for patient_id in patient_to_label.keys():
            if patient_id in base_name:
                matched_patient_id = patient_id
                break

        if matched_patient_id is None:
            print(f"No Patient ID match found for image: {file_name}")
            unmatched_count += 1
            continue

        label = patient_to_label[matched_patient_id]

        class_name = str(label)

        data_by_class[class_name].append(
            (image_path, base_name, class_name)
        )

        matched_count += 1

    print(f"Matched images: {matched_count}")
    print(f"Unmatched images: {unmatched_count}")

    return data_by_class

def stratified_split(data_by_class):
    train_data = []
    val_data = []
    test_data = []

    for class_name, samples in data_by_class.items():
        random.shuffle(samples)

        n = len(samples)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)

        train_samples = samples[:n_train]
        val_samples = samples[n_train:n_train + n_val]
        test_samples = samples[n_train + n_val:]

        train_data.extend(train_samples)
        val_data.extend(val_samples)
        test_data.extend(test_samples)

        print(
            f"{class_name}: total={n}, "
            f"train={len(train_samples)}, "
            f"val={len(val_samples)}, "
            f"test={len(test_samples)}"
        )

    return train_data, val_data, test_data


def save_classification_split(data, split_name):
    for image_path, base_name, class_name in data:
        class_out_dir = os.path.join(target_path, split_name, class_name)
        os.makedirs(class_out_dir, exist_ok=True)

        image_dst = os.path.join(class_out_dir, base_name + ".png")

        process_and_save(image_path, image_dst)


data_by_class = collect_classification_images()

train_data, val_data, test_data = stratified_split(data_by_class)

save_classification_split(train_data, "train")
save_classification_split(val_data, "val")
save_classification_split(test_data, "test")

print("\nClassification dataset created successfully!")

CSV columns:
['Unnamed: 0', 'Patient ID', 'age', 'Sex', 'Sex.1', 'CVD Symptoms', 'packyears', 'smoking', 'Hypertension', 'Diabetes', 'BMI', 'Glucose', 'Tchol', 'HDL', 'LDL', 'Trigl', 'Creat', 'Apoa1', 'ApoB', 'Base CVE', 'FUPEvents', 'TimetoEvent']
Total patients in CSV: 689
No Patient ID match found for image: clin_0701_L.tiff
No Patient ID match found for image: clin_0858_R.tiff
No Patient ID match found for image: clin_0918_R.tiff
No Patient ID match found for image: clin_0885_L.tiff
No Patient ID match found for image: clin_0979_R.tiff
No Patient ID match found for image: clin_1081_L.tiff
No Patient ID match found for image: clin_0944_R.tiff
No Patient ID match found for image: clin_1051_R.tiff
No Patient ID match found for image: clin_0869_R.tiff
No Patient ID match found for image: clin_0705_R.tiff
No Patient ID match found for image: clin_0946_L.tiff
No Patient ID match found for image: clin_0897_L.tiff
No Patient ID match found for image: clin_0975_R.tiff
No Patient ID match fo